# Mi proyecto · De la teoría a la práctica (Sesiones 1 a 3)
**Avance 1 del proyecto integrador** · Módulo: Preparación, Recopilación y Análisis Exploratorio de Datos · MIADAS – UMSA

**Grupo:** Grupo 6 · **Integrantes:**
 - Rayssa Aguirre
 - Marcelo De La Quintana
 - Nancy Illanes Castillo
 - Franklin Fernando Poma Limachi

**Conjunto de datos:** Cooperativa de Ahorros (clientes, cuentas y movimientos) · **Fuente y URL:** Clases Personales, material de práctica — <https://clasespersonales.com/bcr.html>

Este notebook repite, con nuestro propio conjunto de datos, los pasos del caso guiado Olist. Evolucionará en las próximas sesiones con la limpieza, el análisis exploratorio, la normalización y la calidad.

### La pregunta de negocio que guía la práctica
> **¿Qué perfil de cliente (departamento, profesión, antigüedad y número de cuentas) concentra la actividad de la cooperativa, y la presencia en cada departamento guarda relación con su población?**

La primera parte se responde en S3 con la tabla integrada. La segunda necesita una fuente externa (población por departamento del INE) y queda planteada para las próximas sesiones.

### Requisitos mínimos del conjunto de datos
- [x] Al menos **1 000 registros** → 4 001 movimientos sobre 1 000 cuentas de 654 clientes
- [x] Al menos **8 variables** que mezclen escalas → 8 originales + 2 derivadas (nominal, ordinal, intervalo y razón)
- [x] Al menos **una variable de fecha** → `FECHA` (del movimiento) y `FAPERTURA` (apertura de la cuenta)
- [x] **Problemas de calidad** reales → codificación de caracteres dañada en los nombres, fechas leídas como texto, códigos leídos como números, espacios dobles, montos en cero
- [x] **Dos o más tablas** que se puedan integrar → 5 tablas relacionadas por claves
- [ ] **Licencia** → el sitio no publica una licencia explícita; los archivos se ofrecen como material de práctica de un curso (ver el informe de cierre)

## Paso 0 · Cargar los datos
Los archivos son CSV separados por **punto y coma** y se leen directamente desde su URL, por lo que el notebook se ejecuta igual en Colab o en una computadora, sin subir archivos.

La tabla principal (`datos`) es **movimientos**: cada fila es una transacción sobre una cuenta de ahorro.

In [1]:
import math
import pandas as pd

BASE = "https://clasespersonales.com/325/"
datos = pd.read_csv(BASE + "movimientos.csv", sep=";")

print(datos.shape)
datos.head()

(4001, 3)


,CUENTA,MONTO,FECHA
0,8-107-82012-8,789,28/06/2023
1,4-107-12080-3,1253,23/05/2023
2,9-105-75927-5,540,06/04/2023
3,8-101-76395-2,781,06/12/2023
4,9-107-54072-8,207,25/01/2023


Las otras cuatro tablas del mismo caso se cargan aquí para revisar sus tipos en S1 e integrarlas en S3.

`clientes.csv` **no se puede leer con la codificación por defecto** (UTF-8): contiene bytes no válidos en los nombres con Í y Ñ. Se lee con `latin-1` para que cargue sin errores y los caracteres dañados se reparan en S1.2.

In [2]:
try:
    pd.read_csv(BASE + "clientes.csv", sep=";")
except UnicodeDecodeError as error:
    print("Lectura por defecto de clientes.csv falla →", error)

cuentas       = pd.read_csv(BASE + "cuentas.csv", sep=";")
clientes      = pd.read_csv(BASE + "clientes.csv", sep=";", encoding="latin-1")
profesiones   = pd.read_csv(BASE + "profesiones.csv", sep=";")
departamentos = pd.read_csv(BASE + "departamentos.csv", sep=";")

tablas = {"movimientos": datos, "cuentas": cuentas, "clientes": clientes,
          "profesiones": profesiones, "departamentos": departamentos}
pd.DataFrame({
    "filas": {nombre: len(t) for nombre, t in tablas.items()},
    "columnas": {nombre: ", ".join(t.columns) for nombre, t in tablas.items()},
})

Lectura por defecto de clientes.csv falla → 'utf-8' codec can't decode byte 0x81 in position 26: invalid start byte


,filas,columnas
movimientos,4001,"CUENTA, MONTO, FECHA"
cuentas,1000,"CUENTA, CARNET, FAPERTURA"
clientes,654,"CARNET, NOMBRES, CODPROF, CODEPTO"
profesiones,6,"CODPROF, DESCRIPCION"
departamentos,9,"CODEPTO, DESCRIPCION"


### Paso 0 (opcional) · Si tu conjunto de datos está en Kaggle
**No aplica:** los datos no están en Kaggle; se leen por su URL pública en la celda anterior.

---
# S1 · Tipos de datos y escalas (Sesión 1)
**Recordemos:** Stevens (1946) propuso cuatro escalas — **nominal, ordinal, intervalo y razón** — y la escala determina qué cálculos son válidos.

### S1.1 · Tipos de dato
¿Qué columnas se leyeron con un tipo incorrecto (fechas como texto, códigos como números)?

In [3]:
for nombre, tabla in tablas.items():
    print(f"── {nombre} ──")
    tabla.info()
    print()

── movimientos ──
<class 'pandas.DataFrame'>
RangeIndex: 4001 entries, 0 to 4000
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   CUENTA  4001 non-null   str  
 1   MONTO   4001 non-null   int64
 2   FECHA   4001 non-null   str  
dtypes: int64(1), str(2)
memory usage: 93.9 KB

── cuentas ──
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CUENTA     1000 non-null   str  
 1   CARNET     1000 non-null   int64
 2   FAPERTURA  1000 non-null   str  
dtypes: int64(1), str(2)
memory usage: 23.6 KB

── clientes ──
<class 'pandas.DataFrame'>
RangeIndex: 654 entries, 0 to 653
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   CARNET   654 non-null    int64
 1   NOMBRES  654 non-null    str  
 2   CODPROF  654 non-null    int64
 3   CODEPTO  654 non-null    

In [4]:
# Faltantes y filas duplicadas por tabla
pd.DataFrame({
    "valores faltantes": {nombre: int(t.isna().sum().sum()) for nombre, t in tablas.items()},
    "filas duplicadas": {nombre: int(t.duplicated().sum()) for nombre, t in tablas.items()},
})

,valores faltantes,filas duplicadas
movimientos,0,0
cuentas,0,0
clientes,0,0
profesiones,0,0
departamentos,0,0


**Qué se leyó mal:**

| Columna (tabla) | Se leyó como | Debería ser | Motivo |
|---|---|---|---|
| `FECHA` (movimientos) | texto | fecha | viene como `dd/mm/aaaa` |
| `FAPERTURA` (cuentas) | texto | fecha | viene como `dd/mm/aaaa` |
| `CARNET` (cuentas, clientes) | entero | texto | es un identificador: no tiene sentido sumarlo ni promediarlo |
| `CODPROF`, `CODEPTO` (clientes y catálogos) | entero | texto | son códigos de catálogo, no cantidades |
| `NOMBRES` (clientes) | texto con caracteres dañados | texto legible | la Í y la Ñ llegan como bytes no imprimibles |

`CUENTA` (texto) y `MONTO` (entero) se leyeron correctamente. No hay valores faltantes ni filas duplicadas exactas en ninguna tabla.

### S1.2 · Corrige los tipos
Fechas con `pd.to_datetime` (indicando el formato día/mes/año para que no se confundan día y mes) y códigos a texto con `astype(str)`.

In [5]:
# Fechas: de texto dd/mm/aaaa a fecha
datos["FECHA"] = pd.to_datetime(datos["FECHA"], format="%d/%m/%Y")
cuentas["FAPERTURA"] = pd.to_datetime(cuentas["FAPERTURA"], format="%d/%m/%Y")

# Identificadores y códigos: de número a texto
cuentas["CARNET"] = cuentas["CARNET"].astype(str)
clientes["CARNET"] = clientes["CARNET"].astype(str)
for tabla in (clientes, profesiones):
    tabla["CODPROF"] = tabla["CODPROF"].astype(str)
for tabla in (clientes, departamentos):
    tabla["CODEPTO"] = tabla["CODEPTO"].astype(str)

print(datos.dtypes, cuentas.dtypes, clientes.dtypes, sep="\n\n")

CUENTA               str
MONTO              int64
FECHA     datetime64[us]
dtype: object

CUENTA                  str
CARNET                  str
FAPERTURA    datetime64[us]
dtype: object

CARNET     str
NOMBRES    str
CODPROF    str
CODEPTO    str
dtype: object


**Nombres con caracteres dañados.** Antes de reparar, se cuenta cuántos registros están afectados y qué caracteres extraños aparecen.

In [6]:
danados = clientes["NOMBRES"].str.contains(r"[^A-Z .]", regex=True)
dobles = clientes["NOMBRES"].str.contains("  ", regex=False)
print("Clientes con caracteres dañados:", int(danados.sum()), "de", len(clientes))
print("Clientes con espacios dobles:   ", int(dobles.sum()))
print("Caracteres extraños:", sorted({ascii(c) for nombre in clientes.loc[danados, "NOMBRES"] for c in nombre if c not in "ABCDEFGHIJKLMNOPQRSTUVWXYZ ."}))
print("Ejemplos:", [ascii(nombre) for nombre in clientes.loc[danados, "NOMBRES"].head(4)])

Clientes con caracteres dañados: 72 de 654
Clientes con espacios dobles:    4
Caracteres extraños: ["'\\x81'", "'\\x9d'", "'\\xbe'"]
Ejemplos: ["'MENDOZA VDA. DE ORTUNO MAR\\x81A'", "'VASQUEZ QUISBERT MAR\\x81A DEL CARMEN'", "'REGUER\\x81N GUZMAN MAR\\x81A LUISA'", "'PATANA MALDONADO MAR\\x81A PAZ'"]


In [7]:
# Por el contexto (MAR?A, PE?A, QUI?ONES) se deduce el carácter original de cada byte dañado
reparacion = {"\x81": "Í", "\xbe": "Ñ", "\x9d": "Ñ"}
for malo, bueno in reparacion.items():
    clientes["NOMBRES"] = clientes["NOMBRES"].str.replace(malo, bueno, regex=False)

# Espacios dobles o sobrantes
clientes["NOMBRES"] = clientes["NOMBRES"].str.replace(r"\s+", " ", regex=True).str.strip()

print("Quedan dañados:", int(clientes["NOMBRES"].str.contains(r"[^A-ZÑÍ .]", regex=True).sum()))
clientes.loc[danados | dobles].head(8)

Quedan dañados: 0


,CARNET,NOMBRES,CODPROF,CODEPTO
3,2436093,MENDOZA VDA. DE ORTUNO MARÍA,6,4
16,2540249,VASQUEZ QUISBERT MARÍA DEL CARMEN,6,8
20,2545811,REGUERÍN GUZMAN MARÍA LUISA,6,1
21,2548763,PATANA MALDONADO MARÍA PAZ,3,7
43,2744576,SANTIVAÑEZ SALVATIERRA LIDIA,6,7
44,2744705,MARCA DE ALTAMIRANO MARÍA ELENA,2,4
61,2842619,MACHICADO CENTELLAS SOFÍA,6,9
63,2843947,TICONA GUARACHI MARÍA ELENA,6,8


**Variable derivada `MES`.** Se extrae de la fecha el mes del movimiento: es una variable **ordinal** (enero < febrero < … < diciembre) que se usará como estrato en S2.

In [8]:
datos["MES"] = datos["FECHA"].dt.month
print("Periodo de los movimientos:", datos["FECHA"].min().date(), "→", datos["FECHA"].max().date())
print("Aperturas de cuenta:       ", cuentas["FAPERTURA"].min().date(), "→", cuentas["FAPERTURA"].max().date())
datos.head()

Periodo de los movimientos: 2023-01-01 → 2023-12-31
Aperturas de cuenta:        2001-01-03 → 2019-12-29


,CUENTA,MONTO,FECHA,MES
0,8-107-82012-8,789,2023-06-28,6
1,4-107-12080-3,1253,2023-05-23,5
2,9-105-75927-5,540,2023-04-06,4
3,8-101-76395-2,781,2023-12-06,12
4,9-107-54072-8,207,2023-01-25,1


### S1.3 · Escala de cada variable

| Variable | Tabla | Escala | Cálculos válidos |
|---|---|---|---|
| `CUENTA` | movimientos, cuentas | Nominal (identificador) | Conteo, valores únicos, moda |
| `CARNET` | cuentas, clientes | Nominal (identificador) | Conteo, valores únicos |
| `NOMBRES` | clientes | Nominal | Conteo, valores únicos |
| `CODPROF` → profesión | clientes, profesiones | Nominal | Frecuencias, proporciones, moda |
| `CODEPTO` → departamento | clientes, departamentos | Nominal | Frecuencias, proporciones, moda |
| `MES` (derivada de `FECHA`) | movimientos | Ordinal | Frecuencias, moda, mediana, percentiles |
| `FECHA` | movimientos | Intervalo | Mínimo, máximo, diferencias entre fechas (no razones) |
| `FAPERTURA` | cuentas | Intervalo | Mínimo, máximo, diferencias entre fechas |
| `MONTO` | movimientos | Razón | Todos: suma, media, desviación, cocientes (cero real) |
| `ANTIGUEDAD` (derivada en S3: años entre apertura y movimiento) | integrada | Razón | Todos: media, desviación, cocientes |

Los códigos `CODPROF` y `CODEPTO` son números, pero su escala es **nominal**: el departamento 8 no es «el doble» del 4.

In [9]:
# Frecuencias de variables nominales (con la descripción del catálogo)
profesion = clientes["CODPROF"].map(profesiones.set_index("CODPROF")["DESCRIPCION"])
departamento = clientes["CODEPTO"].map(departamentos.set_index("CODEPTO")["DESCRIPCION"])

print(profesion.value_counts().to_string(), "\n")
print(departamento.value_counts().to_string())

CODPROF
ABOGADO        127
ECONOMISTA     121
PROFESOR       119
INFORMATICO    103
INGENIERO       94
CONTADOR        90 

CODEPTO
ORURO         82
COCHABAMBA    82
SANTA CRUZ    80
PANDO         80
TARIJA        76
POTOSI        68
CHUQUISACA    65
BENI          65
LA PAZ        56


In [10]:
# Frecuencias de la variable ordinal MES (en su orden natural)
datos["MES"].value_counts().sort_index()

MES
1     322
2     308
3     339
4     323
5     342
6     304
7     343
8     325
9     351
10    318
11    355
12    371
Name: count, dtype: int64

**Media frente a mediana en una variable ordinal.** `MES` está guardado como número, así que pandas calcula su media sin protestar. ¿Cuál de las dos medidas es adecuada según la escala de Stevens?

In [11]:
print("Media:  ", round(datos["MES"].mean(), 2))
print("Mediana:", datos["MES"].median())
print("Moda:   ", datos["MES"].mode().tolist())

Media:   6.62
Mediana: 7.0
Moda:    [12]


**Lectura:** para una escala **ordinal** lo riguroso es la **mediana** junto con la distribución de frecuencias. La mediana (7) dice que al cierre de julio ya había ocurrido más de la mitad de los movimientos del año, y la moda (12) que diciembre es el mes con más movimientos. La media (6,62) trata el número de mes como una cantidad que se puede sumar y dividir, y devuelve un valor que no corresponde a ningún mes: se puede calcular, pero no se debe interpretar.

In [12]:
# Estadísticos de la variable de razón
datos[["MONTO"]].describe().round(2)

,MONTO
count,4001.00
mean,672.29
std,421.71
min,0.00
25%,325.00
50%,592.00
75%,1016.00
max,1500.00


In [13]:
print("Movimientos con monto 0:         ", int((datos["MONTO"] == 0).sum()))
print("Movimientos con monto menor a 10:", int((datos["MONTO"] < 10).sum()))
print("Movimientos con monto negativo:  ", int((datos["MONTO"] < 0).sum()))

Movimientos con monto 0:          1
Movimientos con monto menor a 10: 13
Movimientos con monto negativo:   0


**Lectura:** los montos van de 0 a 1 500, sin negativos. La tabla no trae una columna que indique si el movimiento es depósito o retiro, así que por ahora `MONTO` solo puede interpretarse como «importe movido». El movimiento con monto 0 y los importes muy pequeños se revisarán en la etapa de limpieza.

---
# S2 · Muestreo (Sesión 2)
**Recordemos:** la fórmula de **Cochran (1977)** con corrección por población finita, y la ventaja del **muestreo estratificado** para representar bien a los grupos pequeños.

**¿Qué subconjunto habría que revisar a mano?** Una auditoría que verifique cada movimiento contra su comprobante no puede revisar los 4 001: se calcula el tamaño de muestra para un 95 % de confianza y 5 % de margen de error, y se compara una muestra aleatoria simple con una estratificada por mes.

In [14]:
def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))

n = tamano_muestra(len(datos))
print(f"Población: {len(datos)} movimientos → muestra: {n} ({n / len(datos):.1%})")
n

Población: 4001 movimientos → muestra: 351 (8.8%)


351

In [15]:
# Sensibilidad: cómo cambia el tamaño de muestra con el margen de error
pd.DataFrame({
    "margen de error": ["10 %", "5 %", "3 %", "1 %"],
    "movimientos (N = 4 001)": [tamano_muestra(len(datos), e) for e in (0.10, 0.05, 0.03, 0.01)],
    "clientes (N = 654)": [tamano_muestra(len(clientes), e) for e in (0.10, 0.05, 0.03, 0.01)],
}).set_index("margen de error")

,movimientos (N = 4 001),clientes (N = 654)
margen de error,,
10 %,94,84
5 %,351,243
3 %,843,406
1 %,2825,613


In [16]:
variable_estrato = "MES"
aleatoria = datos.sample(n=n, random_state=42)
estratificada = datos.groupby(variable_estrato, group_keys=False).sample(frac=n / len(datos), random_state=42)
print("Tamaños → aleatoria:", len(aleatoria), "· estratificada:", len(estratificada))

comparacion = pd.DataFrame({
    "Población": datos[variable_estrato].value_counts(normalize=True),
    "Aleatoria": aleatoria[variable_estrato].value_counts(normalize=True),
    "Estratificada": estratificada[variable_estrato].value_counts(normalize=True),
}).sort_index()
comparacion.round(3)

Tamaños → aleatoria: 351 · estratificada: 352


,Población,Aleatoria,Estratificada
MES,,,
1,0.080,0.105,0.080
2,0.077,0.066,0.077
3,0.085,0.100,0.085
4,0.081,0.074,0.080
5,0.085,0.071,0.085
6,0.076,0.068,0.077
7,0.086,0.088,0.085
8,0.081,0.080,0.082
9,0.088,0.077,0.088


In [17]:
# Desviación máxima respecto a la población (en puntos porcentuales) y comparación del monto
desviacion = comparacion[["Aleatoria", "Estratificada"]].sub(comparacion["Población"], axis=0).abs().max() * 100
print("Desviación máxima de la proporción por mes (puntos porcentuales):")
print(desviacion.round(2).to_string(), "\n")

pd.DataFrame({
    "Población": datos["MONTO"].describe(),
    "Aleatoria": aleatoria["MONTO"].describe(),
    "Estratificada": estratificada["MONTO"].describe(),
}).loc[["count", "mean", "std", "50%"]].round(2)

Desviación máxima de la proporción por mes (puntos porcentuales):
Aleatoria        2.49
Estratificada    0.12 



,Población,Aleatoria,Estratificada
count,4001.00,351.00,352.00
mean,672.29,692.19,695.95
std,421.71,424.24,449.79
50%,592.00,620.00,635.00


**Lectura:**
- Con la corrección por población finita bastan **351 movimientos** (8,8 % del total) para estimar una proporción con ±5 % de error al 95 % de confianza. Si la población fueran los 654 clientes (por ejemplo, para una encuesta de satisfacción), harían falta 243.
- La muestra **estratificada** reproduce casi exactamente el peso de cada mes (desviación máxima de 0,12 puntos porcentuales); la **aleatoria simple** se desvía hasta 2,49 puntos (enero y diciembre quedan sobrerrepresentados). Si lo que se va a revisar depende de la época del año, conviene la estratificada.
- Estratificar por mes garantiza la representatividad **en el mes**, no en las demás variables: el monto medio de ambas muestras (692 y 696) queda algo por encima del poblacional (672), dentro de la variación esperable por azar para una muestra de este tamaño.
- La estratificada tiene 352 filas y no 351 por el redondeo dentro de cada estrato.

---
# S3 · Fuentes e integración (Sesión 3)
**Recordemos:** en una base de datos relacional (**Codd, 1970**) cada tabla guarda una entidad y las tablas se **relacionan por claves**.

El conjunto tiene **cinco tablas** de una misma fuente. Se integran con uniones *left* partiendo de los movimientos, para no perder ninguna transacción:

`movimientos` —`CUENTA`→ `cuentas` —`CARNET`→ `clientes` —`CODPROF`→ `profesiones` · —`CODEPTO`→ `departamentos`

In [18]:
# Antes de unir: ¿las claves son únicas en su tabla y existen en la tabla relacionada?
print("CUENTA única en cuentas:  ", cuentas["CUENTA"].is_unique)
print("CARNET único en clientes: ", clientes["CARNET"].is_unique)
print("Movimientos sin cuenta registrada:    ", int((~datos["CUENTA"].isin(cuentas["CUENTA"])).sum()))
print("Cuentas sin cliente registrado:       ", int((~cuentas["CARNET"].isin(clientes["CARNET"])).sum()))
print("Clientes sin profesión en catálogo:   ", int((~clientes["CODPROF"].isin(profesiones["CODPROF"])).sum()))
print("Clientes sin departamento en catálogo:", int((~clientes["CODEPTO"].isin(departamentos["CODEPTO"])).sum()))

CUENTA única en cuentas:   True
CARNET único en clientes:  True
Movimientos sin cuenta registrada:     0
Cuentas sin cliente registrado:        0
Clientes sin profesión en catálogo:    0
Clientes sin departamento en catálogo: 0


In [19]:
integrado = (
    datos
    .merge(cuentas, on="CUENTA", how="left", validate="many_to_one")
    .merge(clientes, on="CARNET", how="left", validate="many_to_one")
    .merge(profesiones.rename(columns={"DESCRIPCION": "PROFESION"}), on="CODPROF", how="left", validate="many_to_one")
    .merge(departamentos.rename(columns={"DESCRIPCION": "DEPARTAMENTO"}), on="CODEPTO", how="left", validate="many_to_one")
)

# Variable derivada de razón: antigüedad de la cuenta (años) al momento del movimiento
integrado["ANTIGUEDAD"] = ((integrado["FECHA"] - integrado["FAPERTURA"]).dt.days / 365.25).round(1)

print("Antes:", datos.shape, "→ después:", integrado.shape)
print("Faltantes generados por la unión:", int(integrado.isna().sum().sum()))
print("Movimientos anteriores a la apertura de su cuenta:", int((integrado["FECHA"] < integrado["FAPERTURA"]).sum()))
integrado.head()

Antes:

 (4001, 4) → después: (4001, 12)
Faltantes generados por la unión: 0
Movimientos anteriores a la apertura de su cuenta: 0


,CUENTA,MONTO,FECHA,MES,CARNET,FAPERTURA,NOMBRES,CODPROF,CODEPTO,PROFESION,DEPARTAMENTO,ANTIGUEDAD
0,8-107-82012-8,789,2023-06-28,6,3544388,2010-04-05,LORONO DE ESPINOZA JENNY,6,6,PROFESOR,TARIJA,13.2
1,4-107-12080-3,1253,2023-05-23,5,2558304,2003-03-06,USMAYO CHUQUIMIA LOURDES,3,3,INFORMATICO,COCHABAMBA,20.2
2,9-105-75927-5,540,2023-04-06,4,3734302,2012-10-24,CASTILLO CASTILLO MARINA,3,3,INFORMATICO,COCHABAMBA,10.4
3,8-101-76395-2,781,2023-12-06,12,6648881,2008-08-18,TITO PAIRI OBNIBULIA MARTRIANA,2,1,INGENIERO,CHUQUISACA,15.3
4,9-107-54072-8,207,2023-01-25,1,6935512,2019-06-01,MAYTA CUENTAS EMILIANA,5,3,ECONOMISTA,COCHABAMBA,3.7


La unión conserva las 4 001 filas (no duplica ni pierde movimientos) y no genera faltantes: la integridad referencial entre las cinco tablas es completa. Con la tabla integrada ya se pueden responder preguntas que ninguna tabla responde sola.

In [20]:
por_departamento = integrado.groupby("DEPARTAMENTO").agg(
    clientes=("CARNET", "nunique"),
    cuentas=("CUENTA", "nunique"),
    movimientos=("MONTO", "size"),
    monto_total=("MONTO", "sum"),
    monto_medio=("MONTO", "mean"),
).round(1).sort_values("monto_total", ascending=False)
por_departamento["% del monto"] = (por_departamento["monto_total"] / por_departamento["monto_total"].sum() * 100).round(1)
por_departamento

,clientes,cuentas,movimientos,monto_total,monto_medio,% del monto
DEPARTAMENTO,,,,,,
PANDO,80,140,535,356639,666.6,13.3
COCHABAMBA,82,126,516,340073,659.1,12.6
SANTA CRUZ,80,122,501,328854,656.4,12.2
ORURO,82,126,493,325514,660.3,12.1
BENI,65,101,421,292503,694.8,10.9
TARIJA,76,109,431,284925,661.1,10.6
POTOSI,68,95,396,281466,710.8,10.5
CHUQUISACA,65,92,364,249551,685.6,9.3
LA PAZ,56,89,344,230315,669.5,8.6


In [21]:
por_profesion = integrado.groupby("PROFESION").agg(
    clientes=("CARNET", "nunique"),
    movimientos=("MONTO", "size"),
    monto_total=("MONTO", "sum"),
    monto_medio=("MONTO", "mean"),
    antiguedad_media=("ANTIGUEDAD", "mean"),
).round(1).sort_values("monto_total", ascending=False)
por_profesion

,clientes,movimientos,monto_total,monto_medio,antiguedad_media
PROFESION,,,,,
ABOGADO,127,747,507904,679.9,12.2
PROFESOR,119,735,492629,670.2,12.6
ECONOMISTA,121,690,466722,676.4,13.3
CONTADOR,90,627,428210,683.0,12.8
INGENIERO,94,588,399135,678.8,13.5
INFORMATICO,103,614,395240,643.7,13.0


In [22]:
# Concentración: cuentas por cliente y movimientos por cuenta
cuentas_por_cliente = cuentas["CARNET"].value_counts()
print("Cuentas por cliente (n.º de cuentas → n.º de clientes):")
print(cuentas_por_cliente.value_counts().sort_index().to_string(), "\n")
print("Movimientos por cuenta:")
print(datos["CUENTA"].value_counts().describe().round(2).to_string(), "\n")
print("Movimientos que repiten cuenta y día con otro anterior:", int(datos.duplicated(["CUENTA", "FECHA"]).sum()))

Cuentas por cliente (n.º de cuentas → n.º de clientes):
count
1     429
2     165
3      26
4      24
5       2
6       6
9       1
12      1 

Movimientos por cuenta:
count    1000.00
mean        4.00
std         1.55
min         2.00
25%         3.00
50%         3.00
75%         5.00
max        11.00 

Movimientos que repiten cuenta y día con otro anterior: 13


### La pregunta de negocio: ¿qué perfil de cliente concentra la actividad?
Las tablas anteriores muestran que **departamento** y **profesión** reparten la actividad de forma pareja. Falta la cuarta dimensión de la pregunta, el **número de cuentas**. Se marca como *multicuenta* al cliente que tiene más de una cuenta y se compara su peso en clientes con su peso en movimientos y en monto.

In [23]:
cuentas_del_cliente = integrado.groupby("CARNET")["CUENTA"].transform("nunique")
integrado["MULTICUENTA"] = cuentas_del_cliente > 1

perfil = integrado.groupby("MULTICUENTA").agg(
    clientes=("CARNET", "nunique"),
    movimientos=("MONTO", "size"),
    monto_total=("MONTO", "sum"),
    monto_medio=("MONTO", "mean"),
    antiguedad_media=("ANTIGUEDAD", "mean"),
)
for columna in ("clientes", "movimientos", "monto_total"):
    perfil["% " + columna] = perfil[columna] / perfil[columna].sum() * 100
perfil.round(1)

,clientes,movimientos,monto_total,monto_medio,antiguedad_media,% clientes,% movimientos,% monto_total
MULTICUENTA,,,,,,,,
False,429,1729,1154861,667.9,13.1,65.6,43.2,42.9
True,225,2272,1534979,675.6,12.7,34.4,56.8,57.1


In [24]:
# Concentración por cliente: ¿qué parte del monto mueve el 20 % de clientes más activos?
monto_por_cliente = integrado.groupby("CARNET")["MONTO"].sum().sort_values(ascending=False)
top20 = monto_por_cliente.head(round(len(monto_por_cliente) * 0.20))

print("Clientes en el 20 % más activo:", len(top20), "de", len(monto_por_cliente))
print("Porcentaje del monto que mueven:", round(top20.sum() / monto_por_cliente.sum() * 100, 1), "%")

Clientes en el 20 % más activo: 131 de 654
Porcentaje del monto que mueven: 44.1 %


**Resultado:** los clientes multicuenta son el **34,4 %** de los clientes (225 de 654) y concentran el **57,1 %** del monto y el **56,8 %** de los movimientos. El 20 % de clientes más activos (131) mueve el **44,1 %** del monto.

**Lectura:** de las cuatro dimensiones de la pregunta, la única que concentra la actividad es el **número de cuentas**. El importe medio por movimiento es prácticamente el mismo en los dos grupos (667,9 frente a 675,6) y la antigüedad media también (13,1 frente a 12,7 años), así que la diferencia no viene de operaciones más grandes ni de cuentas más antiguas, sino de que quien tiene más cuentas hace más movimientos. La segunda parte de la pregunta (relación con la población de cada departamento) queda pendiente de integrar la fuente externa que se describe a continuación.

**Fuente externa complementaria (para próximas sesiones).** La tabla integrada solo describe a la cooperativa. Para comparar por departamento en términos relativos se podría añadir, usando `DEPARTAMENTO` como clave, la **población por departamento del INE** (Censo de Población y Vivienda) y así calcular clientes o monto movido por cada 100 000 habitantes; y las estadísticas de depósitos del sistema financiero que publica la **ASFI** como referencia de mercado. La unión exigiría antes homologar la escritura de los nombres (aquí están en mayúsculas y sin tildes: `POTOSI`).

---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)
La guía CRISP-DM (Chapman et al., 2000) pide documentar la tarea **«Recolectar los datos iniciales»**. Este es el informe de nuestro conjunto de datos:

| Elemento | Mi conjunto de datos |
|---|---|
| **Fuente** | Sitio «Clases Personales» (clasespersonales.com), material de práctica de su programa de capacitación en R, caso «Cooperativa de Ahorros». El sitio no publica una licencia explícita: los archivos se ofrecen para descarga libre como material de curso y aquí se usan solo con fin académico. Por su origen parecen datos de práctica, pero incluyen nombres y números de carnet, así que se tratan como datos personales. |
| **Tipo de fuente** (interna / externa · primaria / secundaria) | Externa y secundaria: no la generó el grupo y se publicó con otro propósito (enseñanza). Para la cooperativa sería una fuente interna, de tipo transaccional. |
| **Formato y acceso** (CSV, Excel, API… · URL) | 5 archivos CSV separados por punto y coma, leídos directamente por URL: `https://clasespersonales.com/325/` + `movimientos.csv`, `cuentas.csv`, `clientes.csv`, `profesiones.csv`, `departamentos.csv`. Página índice: <https://clasespersonales.com/bcr.html>. `clientes.csv` no está en UTF-8. |
| **Tablas y número de registros** | movimientos: 4 001 × 3 · cuentas: 1 000 × 3 · clientes: 654 × 4 · profesiones: 6 × 2 · departamentos: 9 × 2. Tabla integrada: 4 001 filas × 12 columnas, sin pérdida ni duplicación de filas. |
| **Periodo que cubre** | Movimientos del 01/01/2023 al 31/12/2023 (un año completo). Cuentas abiertas entre el 03/01/2001 y el 29/12/2019. |
| **Problemas encontrados** | **(1) Codificación:** `clientes.csv` falla al leerse en UTF-8 y 72 nombres (11 %) traían la Í y la Ñ como bytes dañados; se repararon. **(2) Tipos:** las dos fechas llegaron como texto y los identificadores (`CARNET`, `CODPROF`, `CODEPTO`) como números; se corrigieron. **(3)** 4 nombres con espacios dobles; se corrigieron. **(4) Escritura inconsistente:** conviven apellidos con y sin Ñ (`NUÑEZ` / `NUNEZ`, `IBAÑEZ` / `IBANEZ`) y posibles errores de tipeo (`NARÍA`); pendiente para la limpieza. **(5)** 1 movimiento con monto 0 y 13 con monto menor a 10. **(6) Variable ausente:** no se indica si el movimiento es depósito o retiro, por lo que no se puede calcular saldos. **(7)** 13 movimientos repiten cuenta y día con otro (a revisar si son legítimos). **(8) Concentración atípica:** un cliente tiene 12 cuentas y otro 9. No hay valores faltantes, filas duplicadas exactas ni claves huérfanas. |
| **Primer hallazgo o pregunta de negocio** | **Hallazgo:** la actividad se concentra por **número de cuentas**: el 34,4 % de los clientes (225 de 654) tiene más de una cuenta y mueve el 57,1 % del monto, y el 20 % de clientes más activos mueve el 44,1 %. Por departamento y profesión, en cambio, ningún segmento se distingue todavía con claridad. El importe medio por movimiento es casi el mismo en todos los departamentos (656 a 711) y profesiones (644 a 683), así que las diferencias en el monto total reflejan la **cantidad de clientes y movimientos**, no el tamaño de cada operación. Esas cantidades tampoco varían mucho: entre 56 y 82 clientes por departamento y entre 304 y 371 movimientos por mes. Son diferencias pequeñas, que pueden deberse al azar; antes de hablar de estacionalidad o de departamentos «fuertes» habrá que contrastarlas en el análisis exploratorio. Un reparto tan parejo es, además, coherente con que se trate de datos de práctica. Un dato sí llama la atención: La Paz es el departamento con **menos** clientes (56), pese a ser uno de los más poblados del país. **Pregunta de negocio:** ¿qué perfil de cliente (departamento, profesión, antigüedad y número de cuentas) concentra la actividad de la cooperativa, y la presencia en cada departamento guarda relación con su población? |

**Tamaño de muestra (S2):** para revisar movimientos a mano con 95 % de confianza y ±5 % de error bastan 351 de los 4 001; para encuestar clientes, 243 de los 654.

**Próximos pasos:** limpieza (homologar la escritura de los nombres, decidir qué hacer con los montos en cero), análisis exploratorio con gráficos por mes, departamento y profesión, e integración con la población por departamento del INE.

---
**Referencias:** Chapman, P. et al. (2000). *CRISP-DM 1.0*. SPSS. · Clases Personales (s. f.). *Caso «Cooperativa de Ahorros»* [conjunto de datos]. <https://clasespersonales.com/bcr.html> · Cochran, W. G. (1977). *Sampling techniques* (3.ª ed.). Wiley. · Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM*, 13(6), 377–387. · Stevens, S. S. (1946). On the theory of scales of measurement. *Science*, 103(2684), 677–680.